# Perfilado de fuentes de ventas

Carga las fuentes sin transformarlas y revisa calidad básica, claves y consistencia entre tablas.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Permite ejecutar el notebook desde la raíz del proyecto o desde src/etl.
current = Path.cwd()
project_root = next(
        {
            "metadata": {
                "language": "markdown"
            },
            "source": [
                "# Perfilado de fuentes de ventas\n\nCarga las fuentes sin transformarlas y revisa calidad básica, claves y consistencia entre tablas.\n"
            ]
        },
    (path for path in [current, *current.parents]
     if (path / 'data' / 'reference_data.json').exists()),
    current,
)
sys.path.insert(0, str(project_root / 'src'))

from etl.extract import extract_from_csv, extract_from_json
from etl.profile import profile_data

In [2]:
sales = next(iter(extract_from_csv(project_root / 'data' / 'sales_transactions.csv').values()))
references = extract_from_json(project_root / 'data' / 'reference_data.json')

print(f'Ventas: {sales.shape[0]:,} filas, {sales.shape[1]} columnas')
print('Referencias:', {name: frame.shape for name, frame in references.items()})

Ventas: 1,000 filas, 9 columnas
Referencias: {'products': (20, 6), 'stores': (3, 5), 'channels': (2, 2), 'promotions': (4, 3)}


In [3]:
report = profile_data(sales, references)

profile_rows = []
for table, values in report['tables'].items():
    profile_rows.append({
        'table': table,
        'rows': values['rows'],
        'duplicate_rows': values['duplicate_rows'],
        'missing_values': sum(values['missing_values'].values()),
        'duplicate_keys': values['duplicate_keys'],
    })
display(__import__('pandas').DataFrame(profile_rows))

print('Referencias inválidas:', report['findings']['invalid_references'])
print('Violaciones de dominio:', report['findings']['domain_violations'])
print('Inconsistencias entre campos:', report['findings']['cross_field_inconsistencies'])

,table,rows,duplicate_rows,missing_values,duplicate_keys
0,sales,1000,0,0,{'sale_line_id': 0}
1,products,20,0,0,{'product_id': 0}
2,stores,3,0,0,{'store_id': 0}
3,channels,2,0,0,{'channel_id': 0}
4,promotions,4,0,0,{'promotion_id': 0}


Referencias inválidas: {}
Violaciones de dominio: {'quantity_non_positive': 0, 'unit_price_sale_non_positive': 0}
Inconsistencias entre campos: {'store_channel_mismatch': 0}
